# WTI Crude Oil — Adaptive Agent Training (Notebook 5 of 7)

> **Part 5 of 7.** Builds on the stateless backtest in [`04_systematic_backtest_eval.ipynb`](04_systematic_backtest_eval.ipynb).

Every method in Notebook 4 was **stateless** — configured once and run.  
This notebook introduces an agent that is different: it has a **training phase**.

The paradigm shift: instead of configuring a model, we onboard an analyst.  
We give the analyst historical performance data to study, let it draw conclusions
and update its own strategy, then put it on live forecasting duty in Notebook 6.

The training paradigm is **curriculum learning** — not time-travel simulation.  
We prepare structured learning material (backtest reports, pre-cached news context)
and hand it to the agent for reflection. The agent decides what to record, based
on the evidence governance rules in its `meta-learning` skill.

**Two training variants are produced:**

| Variant | Strategy dir | Training material |
|---------|-------------|------------------|
| Stats-only | `wti-strategy-stats/` | Activity 1 (exploration) + Activity 2a (backtest report) |
| News-grounded | `wti-strategy-news/` | Activity 2b (same report + weekly news context) |

---
## 0. Setup

In [ ]:
import asyncio
import warnings
from datetime import date
from pathlib import Path

import pandas as pd
import yaml
from IPython.display import Markdown, display

from aieng.forecasting.evaluation.backtest import BacktestResult
from aieng.forecasting.methods.agentic import (
    build_adk_agent,
    build_curriculum_prompt,
    format_backtest_report,
    load_context_documents,
)
from aieng.forecasting.methods.agentic.adk_runner import AdkTextRunner, AdkTextRunnerConfig
from energy_oil_forecasting.adaptive_agent import (
    build_wti_adaptive_config,
)
from energy_oil_forecasting.adaptive_agent.curriculum.snapshot_utils import (
    restore_state,
    snapshot_state,
)
from energy_oil_forecasting.data import WTI_SERIES_ID, build_wti_service

warnings.filterwarnings('ignore')

# ── Paths ─────────────────────────────────────────────────────────────────────
_NB_DIR = Path('.')
_SKILLS_ROOT = _NB_DIR / 'adaptive_agent' / 'skills'
_CURRICULUM_DIR = _NB_DIR / 'adaptive_agent' / 'curriculum'
_CONTEXT_DIR = _CURRICULUM_DIR / 'context'

STATS_STRATEGY_DIR = _SKILLS_ROOT / 'wti-strategy-stats'
NEWS_STRATEGY_DIR  = _SKILLS_ROOT / 'wti-strategy-news'

# ── Model ─────────────────────────────────────────────────────────────────────
AGENT_MODEL = 'gemini-3.1-flash-preview'

# ── Run guards ────────────────────────────────────────────────────────────────
# Expensive activities default to False (outputs committed after first run).
# Set True only when you want to regenerate outputs from scratch.
RUN_ACTIVITY_1  = False   # Agent-initiated code-execution exploration
RUN_ACTIVITY_2A = False   # Statistics-only curriculum delivery
RUN_ACTIVITY_2B = False   # News-grounded curriculum delivery

# ── Data service ──────────────────────────────────────────────────────────────
data_service = build_wti_service()
print('Setup complete.')

---
## 1. Initial State & Pre-Training Snapshot

Before any training activity, we snapshot the clean initial strategy state  
in both variant directories. The reset cell at the end of this notebook  
restores from these snapshots, so you can re-run training from scratch  
without re-running Notebook 4.

In [ ]:
# Snapshot both variant dirs (no-op if snapshot already exists)
print('Snapshotting initial strategy states...')
snapshot_state(STATS_STRATEGY_DIR)
snapshot_state(NEWS_STRATEGY_DIR)
print()

# Show the initial wti-strategy-stats state
print('Initial wti-strategy-stats/SKILL.md:')
print('─' * 60)
print((STATS_STRATEGY_DIR / 'SKILL.md').read_text())

---
## 2. Activity 1 — Agent-Initiated Exploration

We give the adaptive agent a structured analytical question and access to  
historical WTI price data via code execution. The agent fetches the data,  
analyzes vol regime distribution and forecast error patterns for a  
trend-projection approach, then decides whether its findings meet the  
evidence threshold in `meta-learning`.

This activity targets `wti-strategy-stats/`. Any observations or hypotheses  
the agent generates here become the starting point for Activity 2a.

> **Run guard:** `RUN_ACTIVITY_1 = False` by default — outputs are committed  
> so the notebook runs reproducibly without real API calls.

In [ ]:
_ACTIVITY_1_PROMPT = (
    'You have access to historical WTI crude oil price data via run_code. '
    'Please do the following:\n\n'
    '1. Fetch the daily WTI close price series for the full year 2025 using '
    'yfinance (ticker: CL=F).\n'
    '2. Compute 21-day rolling realized volatility. Classify each day into a '
    'vol regime: low (<15% annualized), medium (15-30%), elevated (30-50%), '
    'or extreme (>50%).\n'
    '3. Simulate the errors a simple trend-projection forecaster would make '
    'at 5, 10, and 21 business-day horizons during each regime. Approximate '
    'this using the historical return distribution within each regime window.\n'
    '4. Summarize: in which regimes and at which horizons does trend-projection '
    'tend to produce the largest errors? Is there a directional bias?\n\n'
    'Based on your analysis, decide whether any findings meet the evidence '
    'threshold in your meta-learning skill. If they do, record them. '
    'If not, explain what additional evidence you would need.'
)

if RUN_ACTIVITY_1:
    config = build_wti_adaptive_config(
        model=AGENT_MODEL, strategy_dir=STATS_STRATEGY_DIR
    )
    agent = build_adk_agent(config)
    runner = AdkTextRunner(
        agent, config=AdkTextRunnerConfig(app_name='wti_training_act1')
    )
    print('Running Activity 1 (code execution + reflection)...')
    print('This may take several minutes.\n')
    reply_act1 = asyncio.run(runner.run_text_async(_ACTIVITY_1_PROMPT))
    (_CURRICULUM_DIR / 'activity1_response.txt').write_text(
        reply_act1, encoding='utf-8'
    )
    print(reply_act1)
else:
    _f = _CURRICULUM_DIR / 'activity1_response.txt'
    if _f.exists():
        print(_f.read_text())
    else:
        print('[Activity 1 output not yet committed. '
              'Set RUN_ACTIVITY_1 = True and re-run.]')

In [ ]:
print('wti-strategy-stats/SKILL.md after Activity 1:')
print('─' * 60)
print((STATS_STRATEGY_DIR / 'SKILL.md').read_text())

---
## 3. Activity 2a — Statistics-Only Curriculum

We compile the 2025 backtest results from Notebook 4 into a structured  
report and send it to the adaptive agent as a curriculum document.  
The agent reads the per-horizon coverage and MAE tables, identifies  
systematic patterns, and decides what to record.

This variant continues training `wti-strategy-stats/`.

In [ ]:
# ── Load 2025 backtest results saved by NB04 ────────────────────────────────
_backtest_jsons = sorted(_CURRICULUM_DIR.glob('backtest_*.json'))
if not _backtest_jsons:
    raise FileNotFoundError(
        'No backtest result files found in adaptive_agent/curriculum/. '
        'Run 04_systematic_backtest_eval.ipynb first.'
    )

backtest_results = {}
for f in _backtest_jsons:
    name = f.stem.removeprefix('backtest_')
    backtest_results[name] = BacktestResult.model_validate_json(f.read_text())

print(f'Loaded {len(backtest_results)} backtest result(s):')
for name, r in backtest_results.items():
    print(f'  {name}: {len(r.predictions)} predictions, '
          f'mean CRPS = {r.mean_crps:.4f}')

In [ ]:
# ── Build actuals dict (needed by format_backtest_report) ───────────────────
_best_name = min(backtest_results, key=lambda n: backtest_results[n].mean_crps)
_best_result = backtest_results[_best_name]
print(f"Using '{_best_name}' (mean CRPS = {_best_result.mean_crps:.4f}) "
      'as curriculum basis.\n')

actuals: dict[tuple[str, int], float] = {}
for pred in _best_result.predictions:
    horizon = (pred.forecast_date - pred.as_of).days
    series = data_service.get_series(WTI_SERIES_ID, as_of=pred.forecast_date)
    target_ts = pd.Timestamp(pred.forecast_date)
    row = series[series.index == target_ts]
    if not row.empty:
        actuals[(str(pred.as_of.date()), horizon)] = float(row.iloc[0])

print(f'Resolved {len(actuals)} actuals.')

In [ ]:
# ── Format and display the backtest report ───────────────────────────────────
report = format_backtest_report(
    result=_best_result,
    actuals=actuals,
    title=f'2025 WTI Backtest — {_best_name}',
    training_start=date(2025, 1, 1),
    training_end=date(2025, 12, 31),
)
display(Markdown(report))

In [ ]:
_PREAMBLE_2A = (
    'You are reviewing the 2025 WTI forecasting performance of the strongest '
    'stateless predictor from a systematic backtest. Study the per-horizon '
    'coverage and error statistics. Identify systematic patterns — particularly '
    'where coverage deviates from the 80% target or where MAE is unexpectedly '
    'large. Decide whether any findings meet the evidence threshold in your '
    'meta-learning skill, and if so, record them using the appropriate tools.'
)

prompt_2a = build_curriculum_prompt(
    report=report,
    context_documents=[],
    as_of='2025-12-31',
    preamble=_PREAMBLE_2A,
)

if RUN_ACTIVITY_2A:
    config_2a = build_wti_adaptive_config(
        model=AGENT_MODEL, strategy_dir=STATS_STRATEGY_DIR
    )
    agent_2a = build_adk_agent(config_2a)
    runner_2a = AdkTextRunner(
        agent_2a, config=AdkTextRunnerConfig(app_name='wti_training_2a')
    )
    print('Sending statistics-only curriculum...')
    reply_2a = asyncio.run(runner_2a.run_text_async(prompt_2a))
    (_CURRICULUM_DIR / 'activity2a_response.txt').write_text(
        reply_2a, encoding='utf-8'
    )
    print(reply_2a)
else:
    _f = _CURRICULUM_DIR / 'activity2a_response.txt'
    if _f.exists():
        print(_f.read_text())
    else:
        print('[Activity 2a output not yet committed. '
              'Set RUN_ACTIVITY_2A = True and re-run.]')

In [ ]:
print('wti-strategy-stats/SKILL.md after Activity 2a:')
print('─' * 60)
print((STATS_STRATEGY_DIR / 'SKILL.md').read_text())

---
## 4. Activity 2b — News-Grounded Curriculum

Same backtest report as Activity 2a, now augmented with pre-cached weekly  
news summaries from 2025. Each summary was generated by  
`scripts/cache_wti_curriculum_news.py` with strict `cutoff_date` enforcement,  
so it contains only information publicly available on that Monday.

This variant trains `wti-strategy-news/` — a clean initial state.  
The comparison in Section 5 shows what the news context adds.

In [ ]:
# ── Representative news dates — one per month across 2025 ───────────────────
# Selected to cover OPEC+ meeting windows and seasonal demand inflection points.
_CURRICULUM_NEWS_DATES = [
    '2025-01-06',  # start of year
    '2025-02-03',  # pre-OPEC+ ministerial
    '2025-03-03',  # OPEC+ output decision period
    '2025-04-07',  # post-OPEC+ adjustment
    '2025-05-05',  # spring demand season
    '2025-06-09',  # OPEC+ June meeting
    '2025-07-07',  # summer demand peak
    '2025-08-04',  # late-summer
    '2025-09-08',  # OPEC+ September review
    '2025-10-06',  # Q4 demand build
    '2025-11-03',  # OPEC+ November decisions
    '2025-12-08',  # year-end
]

context_docs = load_context_documents(_CONTEXT_DIR, _CURRICULUM_NEWS_DATES)
print(f'Loaded {len(context_docs)} context documents:')
for d, content in context_docs:
    print(f'  {d}: {len(content):,} chars')

In [ ]:
_PREAMBLE_2B = (
    'You are reviewing 2025 WTI forecasting performance alongside weekly market '
    'context summaries from the same period. The backtest report shows '
    'statistical patterns; the context summaries show what information was '
    'available at key dates. Study both together: does the news context help '
    'explain the error patterns? Identify systematic patterns and decide '
    'whether they meet the evidence threshold in your meta-learning skill. '
    'If so, record them using the appropriate tools.'
)

prompt_2b = build_curriculum_prompt(
    report=report,
    context_documents=context_docs,
    as_of='2025-12-31',
    preamble=_PREAMBLE_2B,
)
print(f'Curriculum prompt: {len(prompt_2b):,} chars '
      f'({len(context_docs)} context documents)')

In [ ]:
if RUN_ACTIVITY_2B:
    config_2b = build_wti_adaptive_config(
        model=AGENT_MODEL, strategy_dir=NEWS_STRATEGY_DIR
    )
    agent_2b = build_adk_agent(config_2b)
    runner_2b = AdkTextRunner(
        agent_2b, config=AdkTextRunnerConfig(app_name='wti_training_2b')
    )
    print('Sending news-grounded curriculum...')
    reply_2b = asyncio.run(runner_2b.run_text_async(prompt_2b))
    (_CURRICULUM_DIR / 'activity2b_response.txt').write_text(
        reply_2b, encoding='utf-8'
    )
    print(reply_2b)
else:
    _f = _CURRICULUM_DIR / 'activity2b_response.txt'
    if _f.exists():
        print(_f.read_text())
    else:
        print('[Activity 2b output not yet committed. '
              'Set RUN_ACTIVITY_2B = True and re-run.]')

In [ ]:
print('wti-strategy-news/SKILL.md after Activity 2b:')
print('─' * 60)
print((NEWS_STRATEGY_DIR / 'SKILL.md').read_text())

---
## 5. Side-by-Side Comparison

What did each training variant learn from the same underlying backtest signal?  
The table below counts each learning layer; the rendered SKILL.md files show  
the qualitative content.

In [ ]:
def _load_yaml_state(strategy_dir: Path) -> dict:
    return yaml.safe_load((strategy_dir / 'skill_state.yaml').read_text())

stats_state = _load_yaml_state(STATS_STRATEGY_DIR)
news_state  = _load_yaml_state(NEWS_STRATEGY_DIR)

rows = []
for label, state in [
    ('wti-strategy-stats (Activity 1 + 2a)', stats_state),
    ('wti-strategy-news  (Activity 2b only)', news_state),
]:
    rows.append({
        'Variant': label,
        'Observations':            len(state.get('observations', [])),
        'Hypotheses':              len(state.get('hypotheses', [])),
        'Calibration corrections': len(state.get('calibration_corrections', [])),
    })

df_comparison = pd.DataFrame(rows).set_index('Variant')
print('Training outcomes summary:')
print(df_comparison.to_string())

In [ ]:
print('\n── wti-strategy-stats SKILL.md ──')
print((STATS_STRATEGY_DIR / 'SKILL.md').read_text())
print('\n── wti-strategy-news SKILL.md ──')
print((NEWS_STRATEGY_DIR / 'SKILL.md').read_text())

---
## 6. Reset

Run the cell below to undo all training activities and restore both  
strategy variants to their pre-training state. This lets you re-run  
training from a clean slate without re-running Notebook 4.

In [ ]:
# ── RESET: restore pre-training state ───────────────────────────────────────
# Uncomment and run to undo all training activities:
#
# restore_state(STATS_STRATEGY_DIR)
# restore_state(NEWS_STRATEGY_DIR)
# print('Both strategy variants restored to pre-training snapshots.')